# Lesson 10: Run an experiment

**You will learn:** how to run a study the way a lab does: design the task, invite a subject, attach recording equipment, run it once, and read back what was recorded.
**Requires:** CPU only, no downloads, under a minute.

| Step | In code |
| --- | --- |
| 1. Invite a subject | a `BrainScoreModel` |
| 2. Choose what to show | a `StimulusSet` |
| 3. Describe the task | a `TaskContext` |
| 4. Attach a neural recorder | `RecordNeural(['IT'])` |
| 5. Attach a behavior recorder | `RecordBehavior()` |
| 6. Run | `Experiment(...).run()` |

Each stimulus is presented **once** per condition. The neural recording and the answer come from that same presentation, so the activity you read back is the activity that produced the answer.

| Term | Meaning |
| --- | --- |
| **Protocol** | The procedure. `TaskProtocol` presents the stimuli under the task. |
| **Condition** | A setting being compared, such as `baseline` or `silenced`. |
| **Trial** | One independent repetition of a condition. The subject is reset before and after it. |
| **Tool** | Equipment attached to the run: recorders, and interventions such as `Intervene`. |

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

## 1. Invite a subject

The toy network from earlier lessons, with one region (`IT`, mapped to layer `'2'`), a readout for behavior, and an `intervention` so it can be perturbed.

In [2]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

In [3]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

model = BrainScoreModel(
    identifier='toy-cnn',
    model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn', model=net, preprocessing=preprocess)},
    region_layer_map={'IT': '2'},
    behavioral_readout_layer='4',
    intervention=build_pytorch_ablation_fn(net),
)
sorted(model.out_channels)

['behavior',
 'behavior:choice',
 'behavior:probabilities',
 'behavior:rating',
 'neural:IT',
 'perturbation']

## 2. Choose what to show

In [4]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


## 3. Describe the task

The task says what the subject does. Here it learns *bright* versus *dark* from labeled examples with a readout. For this lesson the same eight images are used to fit and to test; a real study keeps them separate.

Plain viewing is a task too: `TaskContext(task_type='passive')`.

In [5]:
from brainscore_core import TaskContext

task = TaskContext(task_type='probabilities', label_set=['bright', 'dark'],
                   fitting_stimuli=stimuli, prefer_path='readout')

## 4 and 5. Attach the equipment

The recorders say what is measured. `RecordNeural` and `RecordBehavior` ask the subject for their outputs in the same session; leave one off and that output is not requested. `Intervene` silences layer `'0'` in the `silenced` condition only, through the subject's own `intervention`, and undoes it at the end of each trial.

In [6]:
from brainscore_core import Perturbation, Selection, StateChange
from brainscore.experiments import Intervene, RecordBehavior, RecordNeural

silence = StateChange(kind='ablation', target=Selection(layer='0'),
                      perturbation=Perturbation(kind='zero'))
equipment = [RecordNeural(['IT']), RecordBehavior(), Intervene(silence, conditions=['silenced'])]

## 6. Run

In [7]:
import tempfile
from brainscore.experiments import Experiment, TaskProtocol

experiment = Experiment(
    subject=model,
    protocol=TaskProtocol(stimuli, task, conditions=['baseline', 'silenced']),
    tools=equipment,
    output_dir=tempfile.mkdtemp() + '/run',
)
experiment.validate()   # checks subject, channels and tools without running the model
result = experiment.run()
result.manifest['status']

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/2 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/2 [00:00<?, ?it/s]

'complete'

## Read back what was recorded

In [8]:
baseline_it = result.neural('IT', condition='baseline')
silenced_it = result.neural('IT', condition='silenced')
print('IT activity:', baseline_it.shape, '(presentations x units)')
print('mean |IT| baseline:', float(abs(baseline_it).mean()), ' silenced:', float(abs(silenced_it).mean()))

IT activity: (8, 16) (presentations x units)
mean |IT| baseline: 0.018575336784124374  silenced: 0.0


In [9]:
for condition in ('baseline', 'silenced'):
    choices = result.behavior(condition=condition)
    accuracy = (choices.values.argmax(axis=1) == (stimuli['label'] == 'dark').values).mean()
    print(f'{condition:>9}: accuracy {accuracy:.2f}')

 baseline: accuracy 1.00
 silenced: accuracy 0.50


Silencing layer `'0'` removes everything downstream, so IT goes quiet. The readout was fitted on the intact network and then tested on the silenced one, as an animal trained before a lesion is tested after it. To fit it on the silenced network instead (what information survives the lesion), pass `TaskProtocol(..., refit_readout=True)`.

Each recording says how it was made:

In [10]:
silenced_it.attrs['recording']

{'task_type': 'probabilities',
 'path': 'readout',
 'response': None,
 'forward_pass': 'readout extraction',
 'readout_fit': 'intact',
 'condition': 'silenced'}

**Models that answer.** A model that answers with its own `answer` function (a language model generating a reply) is recorded the same way. Its extractors record the forward pass that reads the stimulus and instruction, before any generated token (`RecordNeural(..., during='prompt')`); activity while the reply is generated is not kept.

## Lower-level tools

For anything the recorders do not cover, the same `Experiment` takes other tools:

| Tool | Does |
| --- | --- |
| `RecordInputsOutputs` | Saves every input, output and lifecycle event. |
| `RecordActivity` | Saves raw layer outputs from every forward pass, through `TorchInstrumentation`. |
| `Ablate`, `ScaleActivity`, `PatchActivity` | Change layer outputs directly. |

See [Experiments](../../docs/guides/experiments.md) for these and for replaying a saved run.

## Summary

- An experiment is a subject, a protocol, equipment and an output folder.
- The task says what the subject does (`'passive'` is plain viewing); the recorders say what is measured, and one presentation gives both.
- With a readout, a lesioned condition is tested with the readout fitted on the intact network unless `refit_readout=True`.
- Conditions are compared settings; trials are repetitions with a reset around each.
- `result.neural(region, condition=...)` and `result.behavior(condition=...)` read back assemblies.

You have now seen every core concept. The [experiments](../experiments/) put them to use on scientific questions.